In [1]:
import pandas as pd
import numpy as np

# Danh sách lưu số dòng sau mỗi bước lọc, ghi vào README
log = []
def ghi(buoc, d):
    #Lưu tên bước và số dòng hiện tại 
    log.append((buoc, len(d)))
    print(f"{buoc}: {len(d):,} dòng")

#Đọc file dữ liệu gốc
df = pd.read_csv("../data/raw/data_public.csv")
ghi("Bước 1: Đọc dữ liệu gốc", df)

Bước 1: Đọc dữ liệu gốc: 51,304 dòng


In [2]:
# Bỏ tin trùng và giữ lại nhà riêng
# Chỉ giữ các dòng có loại bất động sản là "Nhà riêng". Do: giá đất, căn hộ, kho xưởng khác nhau rất xa,trộn chung sẽ làm kết quả phân tích bị nhiễu

df = df.drop_duplicates(subset = "Listing ID") # Mỗi tin đăng có một mã danh sách riêng Listing ID
ghi("Bước 2: Bỏ tin trùng", df) # Nếu cùng một mã xuất hiện nhiều lần thì là tin bị chép trùng và chỉ giữ một

df = df[df["Property Type"] == "Nhà riêng"].copy()
#Kiểm tra từng dòng đúng/sai và tạo bản sao riêng sau này sửa sẽ không bị cảnh báo
ghi("Bước 3: Chỉ giữ nhà riêng", df)

Bước 2: Bỏ tin trùng: 51,304 dòng
Bước 3: Chỉ giữ nhà riêng: 8,184 dòng


In [3]:
#Bỏ tin thiếu toạ độ(k tính được khoảng cách)
df = df.dropna(subset=["Latitude", "Longitude"]) #vĩ độ, kinh độ
ghi("Bước 4: Có toạ độ", df)

#Bỏ tạo độ sai(nằm ngoài HCM)
df=df[df["Latitude"].between(10.2, 11.5) & df["Longitude"].between(106.2, 107.4)]
ghi("Bước 5: Toạ độ hợp lệ", df)

# Tọa độ chợ Bến Thành, đổi sang radian (công thức cần radian)
bt_vido = np.radians(10.7725)
bt_kinhdo = np.radians(106.6980)

# Tọa độ từng căn nhà, đổi sang radian
nha_vido = np.radians(df["Latitude"])
nha_kinhdo = np.radians(df["Longitude"])

# Công thức haversine: tính khoảng cách giữa 2 điểm trên mặt cầu
gia_tri_haversine = (np.sin((nha_vido - bt_vido) / 2) ** 2 + np.cos(nha_vido) * np.cos(bt_vido) * np.sin((nha_kinhdo - bt_kinhdo) / 2) ** 2)

# Khoảng cách (km) = 2 x bán kính Trái Đất (6371 km) x arcsin(căn của giá trị trên)
df["Dist_BenThanh_km"] = 2 * 6371 * np.arcsin(np.sqrt(gia_tri_haversine))

# Chỉ giữ nhà trong 30 km (cũng loại tin Bình Dương, Bà Rịa)
df = df[df["Dist_BenThanh_km"] <= 30]
ghi("Bước 6: Trong 30 km quanh Bến Thành", df)

Bước 4: Có toạ độ: 5,137 dòng
Bước 5: Toạ độ hợp lệ: 5,121 dòng
Bước 6: Trong 30 km quanh Bến Thành: 4,671 dòng


In [4]:
# Xem giá, diện tích, khoảng cách trước khi lọc để biết ngưỡng có hợp lý không
df[["Price", "Area", "Dist_BenThanh_km"]].describe().round(1)

,Price,Area,Dist_BenThanh_km
count,4671.0,4671.0,4671.0
mean,21254.0,165.0,12.6
std,309031.6,475.5,6.9
min,55.5,1.0,0.2
25%,4650.0,62.0,8.1
50%,7500.0,90.0,10.3
75%,13500.0,140.0,15.5
max,16000000.0,16979.6,30.0


In [5]:
# Price tính bằng triệu đồng: giữ 500 triệu -> 200 tỷ
# Area tính bằng m2: giữ 10 -> 1.000
GIA_MIN, GIA_MAX = 500, 200000
DT_MIN, DT_MAX = 10, 1000

df = df[df["Price"].between(GIA_MIN, GIA_MAX) & df["Area"].between(DT_MIN, DT_MAX)]
ghi("Bước 7: Giá và diện tích hợp lý", df)

# Bảng tổng hợp số dòng, chép vào README
pd.DataFrame(log, columns=["Bước", "Số dòng"])

Bước 7: Giá và diện tích hợp lý: 4,582 dòng


,Bước,Số dòng
0,Bước 1: Đọc dữ liệu gốc,51304
1,Bước 2: Bỏ tin trùng,51304
2,Bước 3: Chỉ giữ nhà riêng,8184
3,Bước 4: Có toạ độ,5137
4,Bước 5: Toạ độ hợp lệ,5121
5,Bước 6: Trong 30 km quanh Bến Thành,4671
6,Bước 7: Giá và diện tích hợp lý,4582


In [6]:
# Giá trên mỗi m2 (triệu đồng/m2): để so sánh các nhà có diện tích khác nhau
df["Price_per_m2"] = df["Price"] / df["Area"]

# Xem giá/m2: nhỏ nhất, trung vị, lớn nhất
df["Price_per_m2"].describe().round(1)

count    4582.0
mean      104.8
std       100.1
min         2.7
25%        52.8
50%        92.6
75%       131.6
max      3078.6
Name: Price_per_m2, dtype: float64

In [7]:
#loại bỏ những căn có giá/m² bất thường
# Tìm ngưỡng 1% và 99% của giá/m2
nguong_gia_m2_thap, nguong_gia_m2_cao = df["Price_per_m2"].quantile([0.01, 0.99])
print("Ngưỡng:", round(nguong_gia_m2_thap, 1), "đến", round(nguong_gia_m2_cao, 1))

# Chỉ giữ nhà có giá/m2 nằm giữa hai ngưỡng (bỏ giá quá thấp, quá cao do nhập sai)
df = df[df["Price_per_m2"].between(nguong_gia_m2_thap, nguong_gia_m2_cao)].copy()
ghi("Bước 8: Bỏ 1% giá/m2 thấp nhất và cao nhất", df)

Ngưỡng: 13.3 đến 417.8
Bước 8: Bỏ 1% giá/m2 thấp nhất và cao nhất: 4,490 dòng


In [8]:
# Lấy log: làm phân phối giá bớt lệch, mô hình dễ học hơn
df["Log_Price_per_m2"] = np.log(df["Price_per_m2"])

In [9]:
# Đọc danh sách ga metro số 1
bang_ga = pd.read_csv("../data/raw/metro_stations.csv")

# Hàm tính khoảng cách (km) giữa 2 điểm tọa độ
def tinh_khoang_cach_km(vido1, kinhdo1, vido2, kinhdo2):
    v1, k1, v2, k2 = map(np.radians, [vido1, kinhdo1, vido2, kinhdo2])
    gia_tri = np.sin((v2 - v1) / 2) ** 2 + np.cos(v1) * np.cos(v2) * np.sin((k2 - k1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(gia_tri))

# Khoảng cách từ mỗi nhà đến từng ga, lấy ga gần nhất
khoang_cach_den_cac_ga = pd.DataFrame(
    {ga.Station_Name: tinh_khoang_cach_km(df["Latitude"], df["Longitude"], ga.Latitude, ga.Longitude)
     for ga in bang_ga.itertuples()},
    index=df.index)
df["Dist_Metro_km"] = khoang_cach_den_cac_ga.min(axis=1)
df["Dist_Metro_km"].describe().round(2)

count    4490.00
mean       10.43
std         5.75
min         0.09
25%         6.89
50%         9.13
75%        12.72
max        29.29
Name: Dist_Metro_km, dtype: float64

In [10]:
# Chỉ giữ cột cần dùng (bỏ Title, Description, thông tin môi giới (có thể có số điện thoại))
# và 4 cột thiếu trên 50%: Bathrooms, Direction, Alley Width, Road Type
cac_cot_giu = ["Listing ID", "Location", "Latitude", "Longitude", "Price", "Area",  "Price_per_m2", "Log_Price_per_m2", "Width", "Length", "Bedrooms","Floors", "Position", "Dist_BenThanh_km", "Dist_Metro_km"]
df = df[cac_cot_giu].copy()

# Đếm ô trống mỗi cột
df.isna().sum()

Listing ID             0
Location               0
Latitude               0
Longitude              0
Price                  0
Area                   0
Price_per_m2           0
Log_Price_per_m2       0
Width                418
Length              1327
Bedrooms             908
Floors               927
Position            1343
Dist_BenThanh_km       0
Dist_Metro_km          0
dtype: int64

In [11]:
#chia dữ liệu từ thư viện scikit-learn
from sklearn.model_selection import train_test_split

# Chia 80% train, 20% để test, random_state=42 để lần nào cũng chia giống nhau
bang_train, bang_test = train_test_split(df, test_size=0.2, random_state=42)

# Tạo bản sao riêng 
bang_train, bang_test = bang_train.copy(), bang_test.copy()

# kiểm tra
print("Train:", len(bang_train), "| Test:", len(bang_test))

Train: 3592 | Test: 898


In [12]:
# Điền ô trống cột số bằng trung vị của train (dùng chung cho cả test)
for ten_cot in ["Width", "Length", "Bedrooms", "Floors"]:
    trung_vi_train = bang_train[ten_cot].median()
    print(ten_cot, "điền bằng", trung_vi_train)
    bang_train[ten_cot] = bang_train[ten_cot].fillna(trung_vi_train)
    bang_test[ten_cot] = bang_test[ten_cot].fillna(trung_vi_train)

# Cột chữ: ô trống ghi "Không rõ"
bang_train["Position"] = bang_train["Position"].fillna("Không rõ")
bang_test["Position"] = bang_test["Position"].fillna("Không rõ")

# Kiểm tra: số ô còn trống phải bằng 0
print("Ô còn trống:", bang_train.isna().sum().sum(), "(train),", bang_test.isna().sum().sum(), "(test)")

Width điền bằng 4.8
Length điền bằng 18.0
Bedrooms điền bằng 3.0
Floors điền bằng 2.0
Ô còn trống: 0 (train), 0 (test)


In [13]:
# Lưu 3 file: clean_1 (đủ), train, test
bang_day_du = pd.concat([bang_train, bang_test]).sort_index()
bang_day_du.to_csv("../data/clean/clean_1.csv", index=False, encoding="UTF_8")
bang_train.to_csv("../data/clean/train.csv", index=False, encoding="UTF_8")
bang_test.to_csv("../data/clean/test.csv", index=False, encoding="UTF_8")

# Bảng số dòng, chép vào README
pd.DataFrame(log, columns=["Bước", "Số dòng"])

,Bước,Số dòng
0,Bước 1: Đọc dữ liệu gốc,51304
1,Bước 2: Bỏ tin trùng,51304
2,Bước 3: Chỉ giữ nhà riêng,8184
3,Bước 4: Có toạ độ,5137
4,Bước 5: Toạ độ hợp lệ,5121
5,Bước 6: Trong 30 km quanh Bến Thành,4671
6,Bước 7: Giá và diện tích hợp lý,4582
7,Bước 8: Bỏ 1% giá/m2 thấp nhất và cao nhất,4490
